# Metadata and Tags

What an `S3Path` knows about its object (size, ETag, last modified time, version id, user metadata), when it asks S3 for it, and how to read and change the tags of an object.

## Setup

This cell only prepares the notebook: it connects to the S3 bucket that the s3pathlib CI uses and empties the folder of this chapter, so every run starts from the same state. In your own code you only need the two lines marked below, with your own AWS profile and bucket.
`show_error` prints an exception in one line, so the demos can show what goes wrong.

In [1]:
import os

import boto3
from s3pathlib.api import S3Path
from s3pathlib.api import s3_context

boto_ses = boto3.Session(profile_name=os.environ["S3PATHLIB_TEST_AWS_PROFILE"])
s3_context.attach_s3_client(boto_ses.client("s3"))  # <-- in your code
aws_account_id = boto_ses.client("sts").get_caller_identity()["Account"]
bucket = f"{aws_account_id}-{boto_ses.region_name}-s3pathlib-test"  # <-- your bucket

# every demo of this chapter writes under this folder
s3dir_chapter = S3Path(bucket, "projects/s3pathlib/docs/05-Metadata-and-Tags/")
_ = s3dir_chapter.delete()


def show_error(func):
    try:
        func()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

To show when s3pathlib calls AWS, this cell records the name of every S3 API call the client makes (a botocore event hook; nothing you need in your own code).

In [2]:
calls = []
s3_context.s3_client.meta.events.register(
    "before-call.s3", lambda model, **kwargs: calls.append(model.name)
)

## 1. Metadata properties

`size`, `size_for_human`, `etag`, `last_modified_at`, `version_id`, `expire_at` and `metadata` read the metadata of the object. A path that has none yet loads it with one `head_object` call the first time; the next reads cost nothing.

In [3]:
from datetime import datetime, timezone

s3dir = s3dir_chapter / "properties/"
_ = (s3dir / "report.json").write_text(
    '{"total": 42}',
    metadata={"author": "alice"},
    expires_datetime=datetime(2030, 1, 1, tzinfo=timezone.utc),
)

p = S3Path(bucket, s3dir.key, "report.json")  # a new path, no metadata yet
calls.clear()
print(p.size, p.size_for_human, p.etag)
print(p.metadata, p.expire_at, p.version_id)
print(isinstance(p.last_modified_at, datetime))
calls

13 13 B 0f76e9fd645d9902dd96c79bf4d18748
{'author': 'alice'} 2030-01-01 00:00:00+00:00 null
True


['HeadObject']

`version_id` is `"null"` here: that is the version id S3 gives an object written without versioning (see [Versioning](../07-Versioning/index.ipynb)). `response` is the raw `head_object` response the properties read from, as a copy; it never calls AWS. Note that `write_text` does not set a content type, so S3 uses `binary/octet-stream`; pass `content_type="text/plain"` (or `"application/json"`, ...) when it matters, e.g. for browsers and CloudFront.

In [4]:
response = p.response
response["ContentType"], response["ContentLength"]

('binary/octet-stream', 13)

## 2. A snapshot of one version at one moment

What a path knows is a **snapshot**: it does not follow later changes in S3. `head_object()` loads the latest metadata into the path (and returns the response); `clear_cache()` drops the snapshot so the next read loads it again.

In [5]:
_ = p.write_text('{"total": 42, "status": "final"}')  # someone changes the object
print(p.size)  # still the snapshot
_ = p.head_object()
print(p.size)

13
32


In [6]:
_ = p.write_text("{}")
p.clear_cache()
calls.clear()
print(p.size)
calls

2


['HeadObject']

## 3. Where a snapshot comes from

A path gets a snapshot from the call that produced it:

- a **full** snapshot from `head_object` / `get_object`: `head_object()`, `exists()`, `read_bytes()` / `read_text()`, or the first property read;
- a **partial** snapshot from a write or list call: `write_bytes()`, `copy_file()`, `iter_objects()`, ... It only has the fields that call returns.

A write returns a new path with the snapshot of the put call, so the size and the user metadata you just wrote cost no call:

In [7]:
calls.clear()
p_new = (s3dir / "new.txt").write_text("hello", metadata={"source": "docs"})
print(p_new.size, p_new.metadata)
calls

5 {'source': 'docs'}


['PutObject']

A listed path has the size, ETag and last modified time of the list call. A field the list call does not return, such as `metadata`, is loaded with `head_object` **of the same object version**: s3pathlib sends the listed ETag as `IfMatch`, so it never mixes the size of one object with the metadata of another.

In [8]:
calls.clear()
listed = [p for p in s3dir.iter_objects() if p.basename == "new.txt"][0]
print(listed.size)
print(listed.metadata)
calls

5
{'source': 'docs'}


['ListObjectsV2', 'HeadObject']

If the object was replaced between the list call and the read, that is no longer possible, and the read raises `S3ObjectChangedError` instead of returning the metadata of the new object. List again to get a path to the new object.

In [9]:
listed = [p for p in s3dir.iter_objects() if p.basename == "new.txt"][0]
_ = (s3dir / "new.txt").write_text("replaced", metadata={"source": "someone else"})
show_error(lambda: listed.metadata)

S3ObjectChangedError: S3 object 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/05-Metadata-and-Tags/properties/new.txt' changed since its metadata was loaded (ETag "5d41402abc4b2a76b9719d911017c592"), get a new S3Path to read the new object


A path built from another one (`parent`, `/`, `change`, `copy()`, ...) is a new path and has no snapshot. `copy.copy` and `pickle` keep it.

In [10]:
import copy

p = s3dir / "new.txt"
_ = p.head_object()
print(p.response is not None)
print(p.copy().response, p.change(new_ext=".md").response)
print(copy.copy(p).response is not None)

True
None None
True


## 4. User metadata

User metadata (`x-amz-meta-*` headers) is set when the object is written: `write_bytes`, `write_text`, `open`, `upload_file` (`extra_args={"Metadata": ...}`), or a copy. S3 cannot change the metadata of an existing object; to change it, copy the object onto itself with new metadata (see [Copy, Move and Delete](../06-Copy-Move-Delete/index.ipynb)). S3 stores the keys in lower case, so s3pathlib warns about upper case keys:

In [11]:
import warnings

with warnings.catch_warnings(record=True) as records:
    warnings.simplefilter("always")
    p_new = (s3dir / "case.txt").write_text("x", metadata={"Author": "alice"})
print(records[0].category.__name__)
print(p_new.metadata, (s3dir / "case.txt").head_object()["Metadata"])

UserWarning
{'author': 'alice'} {'author': 'alice'}


## 5. Tags

Tags are key-value pairs on an object, used by lifecycle rules, IAM policies and cost reports. Unlike user metadata they can be changed at any time, without writing a new version. Each method returns `(version_id, tags)`; the version id is `None` in a bucket that never had versioning.

`put_tags` replaces all tags, `update_tags` merges like `dict.update`, `get_tags` reads them, and `put_tags({})` removes them all.

In [12]:
p = s3dir / "tagged.txt"
_ = p.write_text("data", tags={"team": "data"})
print(p.get_tags())
print(p.update_tags({"owner": "alice"}))
print(p.put_tags({"stage": "prod"}))
print(p.put_tags({}))

(None, {'team': 'data'})
(None, {'team': 'data', 'owner': 'alice'})
(None, {'stage': 'prod'})
(None, {})


`update_tags` reads the tags, merges and writes them back: two calls, not atomic. A change made by someone else in between is lost. Tags belong to files; a folder or a bucket path raises:

In [13]:
show_error(lambda: s3dir.get_tags())

S3PathIsNotFileError: 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/05-Metadata-and-Tags/properties/' is not a file!


## Common pitfalls

- **A snapshot does not follow S3.** A path that read its size keeps that size after the object changes; call `head_object()` or `clear_cache()` when you need the latest data.
- **Writes return a new path.** The path you wrote to keeps its old snapshot; read the metadata of what you wrote from the returned path.
- **The snapshot never picks the version.** `read_bytes()`, `get_tags()`, `delete()`, ... always work on the latest version unless you pass `version_id`, even on a path that describes an older version (see [Versioning](../07-Versioning/index.ipynb)).
- **Lower case metadata keys.** `{"Author": ...}` comes back as `{"author": ...}`.

## See also

- [List and Filter](../04-List-and-Filter/index.ipynb): filter by the listed `size`, `etag`, `last_modified_at`.
- [Clients and Errors](../10-Clients-and-Errors/index.ipynb): which client loads the metadata of a path listed with `s3_client=...`.